<a href="https://colab.research.google.com/github/lizaasad04/CV-project-Brain-tumor-localization-with-YOLO/blob/main/CV_Eproject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Libraries

In [ ]:
pip install ultralytics

In [ ]:
import os
import pandas as pd
import nibabel as nib
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter, uniform_filter, median_filter
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from skimage.transform import resize
from skimage.feature import canny, graycomatrix, graycoprops
from skimage.filters import sobel
from skimage.morphology import erosion, dilation, opening, closing, disk
from skimage.measure import find_contours, label, regionprops
from scipy.spatial import ConvexHull
from scipy.ndimage import binary_fill_holes
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from ultralytics import YOLO
import cv2
import csv

Importing dataset

In [ ]:
dataset_path = '/content/drive/MyDrive/brats ped dataset 25 samples'
patients = os.listdir(dataset_path)
print(f"Patients found: {patients}")

# ***E-assignment 01***

### Step 1. batch processing

In [ ]:
images = {} #dictionary to store all loaded images

for patient in sorted(os.listdir(dataset_path)):
    patient_path = os.path.join(dataset_path, patient)
    if os.path.isdir(patient_path):
        images[patient] = {}
        for file in sorted(os.listdir(patient_path)):
            if file.endswith('.nii.gz') or file.endswith('.nii'):
              if "seg" in file: #we will skip the segmentation mask wali files
                continue
              modality = file.split('_')[-1].replace('.nii.gz', '').replace('.nii', '')
              filepath = os.path.join(patient_path, file)
              img = nib.load(filepath)
              images[patient][modality] = img.get_fdata()
              print(f"Loaded: {patient} | {modality} | Shape: {img.get_fdata().shape}")

print(f"\nTotal patients loaded: {len(images)}")

### Step 2: Radiometric Resolution (bit-depth detection)

we'll find the bit depth for each of the 4 slices in each of the 25c patient folders

In [ ]:
modalities_to_process = ['t1n','t1c', 't2w', 't2f']

for patient, modalities in images.items(): #going thru every patient and their 4 MRI scans that we loaded in prev step
    print(f"\n{patient}:")
    for modality in modalities_to_process:
      # Construct the correct key for the image data
        image_key = f"{patient}-{modality}"
      #getting basic stats. min_val/max_val means the darkest and brightest pixel values in the scan
        data = modalities[image_key]
        min_val = data.min()
        max_val = data.max()
        unique_vals = len(np.unique(data)) #unique_val is how many distinct intensity values exist in the image

        #determine bit depth
        if max_val <= 255:
            bit_depth = "8-bit"
        elif max_val <= 65535:
            bit_depth = "16-bit"
        else:
            bit_depth = "32-bit float"

        print(f"  {modality} | Min: {min_val:.2f} | Max: {max_val:.2f} | "
              f"Unique values: {unique_vals} | Bit-depth: {bit_depth} | "
              f"Dtype: {data.dtype}")

Determining bit depth:
* 8-bit → pixel values range 0–255 (256 levels)
* 16-bit → pixel values range 0–65535 (65,536 levels)
* 32-bit float → beyond that, floating point precision

false contouring: if we process a 16-bit image as if it were 8-bit,we get False Contouring — artificial bands/edges appearing in smooth gradient areas of the MRI, like a topographic map effect. Knowing the bit-depth upfront prevents that.


### Step 3: Filtering

firstly we will visualise raw slices for all 25 patients  before filtering so we have a "before" ref

In [ ]:
slice_idx = 77  # middle slice

# Calculate the number of rows needed for all patients and adjust figsize dynamically
num_patients = len(images.keys())
num_modalities = len(modalities_to_process)
fig_height = 5 * num_patients  # Adjust height based on number of patients

fig, axes = plt.subplots(num_patients, num_modalities, figsize=(20, fig_height))

for i, patient in enumerate(sorted(images.keys())):
    for j, modality in enumerate(modalities_to_process):
        # Construct the correct key for accessing the image data
        image_data_key = f"{patient}-{modality}"
        raw_slice = images[patient][image_data_key][:, :, slice_idx]
        raw_normalized = (raw_slice - raw_slice.min()) / (raw_slice.max() - raw_slice.min() + 1e-8)

        axes[i][j].imshow(raw_normalized, cmap='gray')
        axes[i][j].set_title(f'{patient}\n{modality.upper()}')
        axes[i][j].axis('off')

plt.suptitle('Original MRI Slices (Before Processing) — All Patients', fontsize=16)
plt.tight_layout()
plt.show()

applying filters : gaussian, mean and median

for each patient, unkay 4 brain mri scans hain. output will show each patients 4 scans along with their mean, median and gussian images after applyinf those filters.

In [ ]:
# Store filtered results for PSNR/SSIM later
filtered_results = {}

slice_idx = 77

for patient in sorted(images.keys()):
    filtered_results[patient] = {}

    fig, axes = plt.subplots(4, 4, figsize=(20, 20))

    for j, modality in enumerate(modalities_to_process):
        # Construct the correct key for accessing the image data
        image_data_key = f"{patient}-{modality}"
        raw_slice = images[patient][image_data_key][:, :, slice_idx]
        raw_normalized = (raw_slice - raw_slice.min()) / (raw_slice.max() - raw_slice.min() + 1e-8)

        # Apply filters
        gaussian_filtered = gaussian_filter(raw_normalized, sigma=1)
        mean_filtered = uniform_filter(raw_normalized, size=3)
        median_filtered = median_filter(raw_normalized, size=3)

        # Store for later
        filtered_results[patient][modality] = {
            'original': raw_normalized,
            'gaussian': gaussian_filtered,
            'mean': mean_filtered,
            'median': median_filtered
        }

        # Plot
        for k, (name, img) in enumerate([('Original', raw_normalized),
                                          ('Gaussian', gaussian_filtered),
                                          ('Mean', mean_filtered),
                                          ('Median', median_filtered)]):
            axes[j][k].imshow(img, cmap='gray')
            axes[j][k].set_title(f'{modality.upper()} - {name}')
            axes[j][k].axis('off')

    plt.suptitle(f'Before/After Filtering — {patient}', fontsize=14)
    plt.tight_layout()
    plt.show()
    print(f"Done: {patient}")

### Step 4: PSNR and SSIM metrics to quantify how well each filter restored the image

the next cells o/p shows the PSNR and SSIM for each patients 4 MRI slices, and further the PSNR and SSIM values for each of gaussian, mean and median filters.

In [ ]:
#calculate PSNR and SSIM for all patients and modalities

print(f"{'Patient':<25} {'Modality':<10} {'Filter':<12} {'PSNR (dB)':<12} {'SSIM':<10}")
print("=" * 70)

for patient in sorted(filtered_results.keys()):
    for modality in modalities_to_process:
        data = filtered_results[patient][modality]
        original = data['original']
        data_range = original.max() - original.min()

        for filter_name in ['gaussian', 'mean', 'median']:
            filtered = data[filter_name]

            psnr_val = psnr(original, filtered, data_range=data_range)
            ssim_val = ssim(original, filtered, data_range=data_range)

            print(f"{patient:<25} {modality:<10} {filter_name:<12} {psnr_val:<12.4f} {ssim_val:<10.4f}")

        print("-" * 70)

PSNR (Peak Signal-to-Noise Ratio)
Measures how much quality was lost after filtering compared to the original.
* Higher = less distortion = filter preserved the image better
* Below 20 dB = bad, 30-40 dB = good, above 40 dB = excellent

SSIM (Structural Similarity Index)
Measures whether the filtered image still looks structurally similar to the original — edges, textures, patterns.

* Ranges from 0 to 1, closer to 1 = better
* Basically asks "does it still look like the same brain?"

### Step 5: Anti-Aliasing Challenge (applying Gaussian pre-filter before downsampling to avoid the Checkerboard Effect)

pehlay patient 26 ka eik slice with and w/o antialisaing, then unka second brain slice then 3rd then 4th. then 2nd patient and so on

In [ ]:
slice_idx = 77

for patient in sorted(images.keys()):
    fig, axes = plt.subplots(4, 3, figsize=(15, 20))

    for j, modality in enumerate(modalities_to_process):
        # Construct the correct key for accessing the image data
        image_data_key = f"{patient}-{modality}"
        raw_slice = images[patient][image_data_key][:, :, slice_idx]
        raw_normalized = (raw_slice - raw_slice.min()) / (raw_slice.max() - raw_slice.min() + 1e-8)

        # Without anti-aliasing
        downsampled_no_aa = resize(raw_normalized, (120, 120), anti_aliasing=False, preserve_range=True)

        # With anti-aliasing (Gaussian pre-filter first)
        gaussian_prefiltered = gaussian_filter(raw_normalized, sigma=1)
        downsampled_with_aa = resize(gaussian_prefiltered, (120, 120), anti_aliasing=True, preserve_range=True)

        axes[j][0].imshow(raw_normalized, cmap='gray')
        axes[j][0].set_title(f'{modality.upper()} - Original (240x240)')
        axes[j][0].axis('off')

        axes[j][1].imshow(downsampled_no_aa, cmap='gray')
        axes[j][1].set_title(f'{modality.upper()} - No Anti-Aliasing (120x120)')
        axes[j][1].axis('off')

        axes[j][2].imshow(downsampled_with_aa, cmap='gray')
        axes[j][2].set_title(f'{modality.upper()} - With Anti-Aliasing (120x120)')
        axes[j][2].axis('off')

    plt.suptitle(f'Anti-Aliasing Challenge — {patient}', fontsize=14)
    plt.tight_layout()
    plt.show()
    print(f"Done: {patient}")

food for thought:

why is it that the pics w/o anti aliasing look sharper and so called "better" as opposed to pics w/ anti aliasing? i thought pics with anti alisiang are better?

* thing is, that sharpness is FAKE. When we downsample from 240×240 to 120×120, we're throwing away half the pixels. Without a prefilter, the algorithm randomly picks which pixels to keep, this creates artificial sharp edges and patterns that weren't in the original image. That "sharpness" is actually errors and artifacts, not real brain detail.

### Saving before and after pics

In [ ]:
# Create output folders
output_path = '/content/drive/MyDrive/CV_Assignment1_Output_final'
os.makedirs(f'{output_path}/before', exist_ok=True)
os.makedirs(f'{output_path}/after/filtered', exist_ok=True)
os.makedirs(f'{output_path}/after/antialiasing', exist_ok=True)

slice_idx = 77

for patient in sorted(images.keys()):
    for modality in modalities_to_process:
        # Construct the correct key for accessing the image data
        image_data_key = f"{patient}-{modality}"
        raw_slice = images[patient][image_data_key][:, :, slice_idx]
        raw_normalized = (raw_slice - raw_slice.min()) / (raw_slice.max() - raw_slice.min() + 1e-8)

        # --- BEFORE ---
        plt.figure(figsize=(6,6))
        plt.imshow(raw_normalized, cmap='gray')
        plt.title(f'Original - {patient} | {modality.upper()}')
        plt.axis('off')
        plt.savefig(f'{output_path}/before/{patient}_{modality}_original.png', bbox_inches='tight')
        plt.close()

        # --- AFTER: Filters ---
        gaussian_filtered = gaussian_filter(raw_normalized, sigma=1)
        mean_filtered = uniform_filter(raw_normalized, size=3)
        median_filtered = median_filter(raw_normalized, size=3)

        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        axes[0].imshow(gaussian_filtered, cmap='gray')
        axes[0].set_title('Gaussian Filter')
        axes[0].axis('off')
        axes[1].imshow(mean_filtered, cmap='gray')
        axes[1].set_title('Mean Filter')
        axes[1].axis('off')
        axes[2].imshow(median_filtered, cmap='gray')
        axes[2].set_title('Median Filter')
        axes[2].axis('off')
        plt.suptitle(f'After Filtering - {patient} | {modality.upper()}')
        plt.tight_layout()
        plt.savefig(f'{output_path}/after/filtered/{patient}_{modality}_filtered.png', bbox_inches='tight')
        plt.close()

        # --- AFTER: Anti-Aliasing ---
        downsampled_no_aa = resize(raw_normalized, (120, 120), anti_aliasing=False, preserve_range=True)
        gaussian_prefiltered = gaussian_filter(raw_normalized, sigma=1)
        downsampled_with_aa = resize(gaussian_prefiltered, (120, 120), anti_aliasing=True, preserve_range=True)

        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        axes[0].imshow(raw_normalized, cmap='gray')
        axes[0].set_title('Original (240x240)')
        axes[0].axis('off')
        axes[1].imshow(downsampled_no_aa, cmap='gray')
        axes[1].set_title('Without Anti-Aliasing (120x120)')
        axes[1].axis('off')
        axes[2].imshow(downsampled_with_aa, cmap='gray')
        axes[2].set_title('With Anti-Aliasing (120x120)')
        axes[2].axis('off')
        plt.suptitle(f'Anti-Aliasing - {patient} | {modality.upper()}')
        plt.tight_layout()
        plt.savefig(f'{output_path}/after/antialiasing/{patient}_{modality}_antialiasing.png', bbox_inches='tight')
        plt.close()

        print(f"Saved: {patient} | {modality}")

print("\nAll images saved to Google Drive successfully!")

# ***E-assignment 02***

### Step 1. Segmentation: Edge Detection (Canny & Sobel)

ok one imp thing, here we wont use sloce 77 anymore. we have a file for every patient called seg, which is segmentation masks, which will tell us the specific area for the tumor. so instead of slice 77, well go with this approach: "Use a different slice where tumor is guaranteed to exist"

In [ ]:
# slice_idx = 77  # removed fixed slice

# Find best slice per patient inside the loop, after loading seg_data:
seg_slice_idx = np.argmax([np.sum(seg_data[:, :, i] > 0) for i in range(seg_data.shape[2])])
output_path = '/content/drive/MyDrive/CV_Assignment2_Output/edge_detection'
os.makedirs(output_path, exist_ok=True)

for patient_id in sorted(images.keys()):
    patient_path = os.path.join(dataset_path, patient_id)

    # Load seg file
    seg_file = [f for f in os.listdir(patient_path) if 'seg' in f][0]
    seg_data = nib.load(os.path.join(patient_path, seg_file)).get_fdata()

    # Find slice with largest tumor area
    seg_slice_idx = np.argmax([np.sum(seg_data[:, :, i] > 0) for i in range(seg_data.shape[2])])

    seg_slice = seg_data[:, :, seg_slice_idx]
    tumor_region = (seg_slice > 0).astype(float)

    for modality in modalities_to_process:
        image_data_key = f'{patient_id}-{modality}'
        raw_slice = images[patient_id][image_data_key][:, :, seg_slice_idx]
        raw_normalized = (raw_slice - raw_slice.min()) / (raw_slice.max() - raw_slice.min() + 1e-8)

        tumor_pixels = raw_normalized * tumor_region

        canny_edges = canny(tumor_pixels, sigma=2)
        sobel_edges = sobel(tumor_pixels)
        sobel_binary = sobel_edges > 0.05

        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        axes[0].imshow(tumor_pixels, cmap='gray')
        axes[0].set_title('Tumor Region')
        axes[0].axis('off')
        axes[1].imshow(canny_edges, cmap='gray')
        axes[1].set_title('Canny Edge Detection')
        axes[1].axis('off')
        axes[2].imshow(sobel_binary, cmap='gray')
        axes[2].set_title('Sobel Edge Detection')
        axes[2].axis('off')

        plt.suptitle(f'Edge Detection — {patient_id} | {modality.upper()} | Slice {seg_slice_idx}', fontsize=14)
        plt.tight_layout()
        plt.savefig(f'{output_path}/{patient_id}_{modality}_edges.png', bbox_inches='tight')
        plt.show()
        plt.close()

    print(f"Done: {patient_id}")

print("\nAll patients saved!")

### Step 2: Morphological Cleaning

This script takes the binary edge masks from the previous step and applies a series of operations to create a solid, clean boundary for the next stage (Chain Coding).

In [ ]:
se = disk(2)
output_path = '/content/drive/MyDrive/CV_Assignment2_Output/morphological'
os.makedirs(output_path, exist_ok=True)

for patient_id in sorted(images.keys()):
    patient_path = os.path.join(dataset_path, patient_id)

    # Load seg file
    seg_file = [f for f in os.listdir(patient_path) if 'seg' in f][0]
    seg_data = nib.load(os.path.join(patient_path, seg_file)).get_fdata()

    # Find best slice
    seg_slice_idx = np.argmax([np.sum(seg_data[:, :, i] > 0) for i in range(seg_data.shape[2])])
    seg_slice = seg_data[:, :, seg_slice_idx]
    tumor_region = (seg_slice > 0).astype(float)

    for modality in modalities_to_process:
        image_data_key = f'{patient_id}-{modality}'
        raw_slice = images[patient_id][image_data_key][:, :, seg_slice_idx]
        raw_normalized = (raw_slice - raw_slice.min()) / (raw_slice.max() - raw_slice.min() + 1e-8)

        # Mask tumor region
        tumor_pixels = raw_normalized * tumor_region

        # Canny then fill
        canny_mask = canny(tumor_pixels, sigma=2)
        binary_mask = binary_fill_holes(canny_mask)

        # Morphological operations
        eroded  = erosion(binary_mask, se)
        dilated = dilation(binary_mask, se)
        opened  = opening(binary_mask, se)
        closed  = closing(binary_mask, se)

        # Plot and save
        fig, axes = plt.subplots(1, 5, figsize=(25, 5))
        for ax, img, title in zip(axes,
            [binary_mask, eroded, dilated, opened, closed],
            ['Filled Mask', 'Erosion', 'Dilation', 'Opening', 'Closing']):
            ax.imshow(img, cmap='gray')
            ax.set_title(title)
            ax.axis('off')

        plt.suptitle(f'Morphological Cleaning — {patient_id} | {modality.upper()} | Slice {seg_slice_idx}', fontsize=14)
        plt.tight_layout()
        plt.savefig(f'{output_path}/{patient_id}_{modality}_morphological.png', bbox_inches='tight')
        plt.show()
        plt.close()

    print(f"Done: {patient_id}")

print("\nAll patients saved!")

### Step 3 and 4: Boundary representation, and Derive first difference and shape number

The Strategy
1. Selection: We will use the Closing mask from the previous step, as it provides the most continuous boundary.
2. Contour Extraction: We use find_contours to get the $(x, y)$ coordinates of the tumor boundary.
3. Encoding: We convert the movement between consecutive coordinates into a sequence of numbers from 0 to 7.


---



*Q) kis basis par brain par particular areas highlight ho rahay hain? likw whats the reasoning behind it?*

 The boundaries are highlighted based on the mathematical detection of Intensity Gradients and the physical properties of the brain tissues captured by the MRI scanner.

REASONING:

1. Mathematical Basis: Intensity Gradients
At its simplest level, the computer does not know what a "tumor" is; it only sees numbers. The Canny and Sobel filters work by calculating the gradient (the rate of change) between neighboring pixels.

* Edges = Sudden Change: If a pixel has a value of 10 (dark) and the pixel next to it has a value of 200 (bright), the computer calculates a high gradient at that point.

* The Threshold: The "highlighting" occurs where these changes are sharpest. The algorithm marks these locations as "edges."

2. Biological Basis: Tissue Contrast
The reason these gradients happen in the first place is due to how different tissues respond to the MRI's magnetic field and pulses.

* Healthy vs. Pathological: Healthy brain tissue (gray matter, white matter, CSF) has very specific, uniform intensity ranges. Tumors, however, often contain more fluid (edema) or have high concentrations of contrast agents (in T1c scans).

* Modalities: * T1c (Contrast Enhanced): Highlights the "active" tumor core because the contrast agent leaks into the tumor through broken blood vessels.

* T2/FLAIR: Highlights the "edema" or swelling around the tumor, which appears very bright compared to the rest of the brain.

3. Structural Basis: Morphological Continuity
The raw edges found by the filters are often messy or "broken." The Morphological Cleaning step (Dilation and Closing) acts as a bridge.

* Grouping: It assumes that if many edge pixels are close together, they belong to the same structure.

* Gap Filling: By thickening these pixels and then thinning them back down, the computer "heals" gaps in the boundary. The final highlighted red line is the result of the computer tracing the outermost perimeter of these connected groups.


---

To ensure our tumor descriptors are robust against the orientation of the patient's head in the MRI scanner (tilted scans), we need to transform the Absolute Chain Code into a Shape Number.

This process involves two mathematical steps: the First Difference (to achieve rotation invariance) and Normalization (to achieve starting-point invariance).


1. The First Difference (Rotation Invariance): The chain code we generated earlier (e.g., $0, 0, 7, 6...$) depends on the absolute orientation of the object. If the tumor rotates, every digit in the code changes.The First Difference counts the number of 45-degree steps required to get from one direction to the next in a counter-clockwise manner.
* Formula: $d_i = (c_i - c_{i-1}) \pmod 8$

* For the first element ($d_1$), we use the last element of the code as $c_{i-1}$ to treat the boundary as a closed loop.

2. The Shape Number (Starting-Point Invariance):
Even with the First Difference, if we start tracing the tumor from a different pixel, the sequence of numbers will be shifted. The Shape Number is defined as the smallest integer magnitude (lexicographical minimum) of all circular shifts of the First Difference.

* By finding the "smallest" version of the code, we ensure that no matter where the computer starts "looking" at the tumor, the resulting descriptor is identical.

In [ ]:
def get_chain_code(contour):
    directions = {
        (0, 1): 0, (-1, 1): 1, (-1, 0): 2, (-1, -1): 3,
        (0, -1): 4, (1, -1): 5, (1, 0): 6, (1, 1): 7
    }
    chain = []
    for i in range(len(contour) - 1):
        dy = int(round(contour[i+1][0] - contour[i][0]))
        dx = int(round(contour[i+1][1] - contour[i][1]))
        dy = max(-1, min(1, dy))
        dx = max(-1, min(1, dx))
        if (dy, dx) in directions:
            chain.append(directions[(dy, dx)])
    return chain

def get_first_difference(chain):
    return [(chain[i+1] - chain[i]) % 8 for i in range(len(chain)-1)]

def get_shape_number(first_diff):
    doubled = first_diff * 2
    min_val = first_diff.copy()
    for i in range(len(first_diff)):
        rotated = doubled[i:i+len(first_diff)]
        if rotated < min_val:
            min_val = rotated
    return min_val

output_path = '/content/drive/MyDrive/CV_Assignment2_Output/chaincode'
os.makedirs(output_path, exist_ok=True)

csv_file = open(f'{output_path}/chaincode_data.csv', 'w', newline='')
writer = csv.writer(csv_file)
writer.writerow(['Patient', 'Modality', 'Slice', 'Chain_Code_First20', 'First_Difference_First20', 'Shape_Number_First20'])

for patient_id in sorted(images.keys()):
    patient_path = os.path.join(dataset_path, patient_id)

    # Load seg file
    seg_file = [f for f in os.listdir(patient_path) if 'seg' in f][0]
    seg_data = nib.load(os.path.join(patient_path, seg_file)).get_fdata()

    # Find best slice
    seg_slice_idx = np.argmax([np.sum(seg_data[:, :, i] > 0) for i in range(seg_data.shape[2])])
    seg_slice = seg_data[:, :, seg_slice_idx]
    tumor_region = (seg_slice > 0).astype(float)

    for modality in modalities_to_process:
        image_data_key = f'{patient_id}-{modality}'
        raw_slice = images[patient_id][image_data_key][:, :, seg_slice_idx]
        raw_normalized = (raw_slice - raw_slice.min()) / (raw_slice.max() - raw_slice.min() + 1e-8)

        # Mask tumor region
        tumor_pixels = raw_normalized * tumor_region

        # Get binary mask
        canny_mask = canny(tumor_pixels, sigma=2)
        binary_mask = binary_fill_holes(canny_mask)

        contours = find_contours(binary_mask, 0.5)
        if len(contours) == 0:
            print(f"No contours: {patient_id} | {modality}")
            continue

        largest = max(contours, key=len)
        chain = get_chain_code(largest)
        if len(chain) < 2:
            continue
        first_diff = get_first_difference(chain)
        shape_num = get_shape_number(first_diff)

        # Save to CSV
        writer.writerow([patient_id, modality, seg_slice_idx, chain[:20], first_diff[:20], shape_num[:20]])

        # Plot
        fig, axes = plt.subplots(1, 2, figsize=(12, 6))
        axes[0].imshow(raw_normalized, cmap='gray')
        axes[0].plot(largest[:, 1], largest[:, 0], 'r-', linewidth=1)
        axes[0].set_title('Tumor Contour')
        axes[0].axis('off')
        axes[1].imshow(binary_mask, cmap='gray')
        axes[1].set_title('Binary Mask')
        axes[1].axis('off')

        plt.suptitle(f'Chain Code — {patient_id} | {modality.upper()} | Slice {seg_slice_idx}', fontsize=14)
        plt.tight_layout()
        plt.savefig(f'{output_path}/{patient_id}_{modality}_chaincode.png', bbox_inches='tight')
        plt.show()
        plt.close()

        print(f"  {modality} | Chain (first 20): {chain[:20]}")
        print(f"  {modality} | First Diff (first 20): {first_diff[:20]}")
        print(f"  {modality} | Shape Num (first 20): {shape_num[:20]}")

    print(f"Done: {patient_id}")

csv_file.close()
print("\nAll patients done! Chain codes saved to CSV.")

### Step 5: computational geometry

For the Computational Geometry stage, we focus on defining the Convex Hull of the tumor. the goal is to find the smallest convex polygon that completely encloses the detected tumor boundary.

In medical terms, the "Convex Hull" helps quantify the solidity of a tumor. A highly irregular tumor (like a Glioblastoma) will have a much larger Convex Hull area compared to its actual area, whereas a smooth, rounded tumor will fit its hull almost perfectly.





---
The output of the Computational Geometry step consists of 25 consolidated image files (one for each patient) that visually prove the computer has successfully "enclosed" the tumor.


In [ ]:
output_path = '/content/drive/MyDrive/CV_Assignment2_Output/convex_hull'
os.makedirs(output_path, exist_ok=True)

for patient_id in sorted(images.keys()):
    patient_path = os.path.join(dataset_path, patient_id)

    # Load seg file
    seg_file = [f for f in os.listdir(patient_path) if 'seg' in f][0]
    seg_data = nib.load(os.path.join(patient_path, seg_file)).get_fdata()

    # Find best slice
    seg_slice_idx = np.argmax([np.sum(seg_data[:, :, i] > 0) for i in range(seg_data.shape[2])])
    seg_slice = seg_data[:, :, seg_slice_idx]
    tumor_region = (seg_slice > 0).astype(float)

    for modality in modalities_to_process:
        image_data_key = f'{patient_id}-{modality}'
        raw_slice = images[patient_id][image_data_key][:, :, seg_slice_idx]
        raw_normalized = (raw_slice - raw_slice.min()) / (raw_slice.max() - raw_slice.min() + 1e-8)

        # Mask tumor region
        tumor_pixels = raw_normalized * tumor_region

        # Get binary mask
        canny_mask = canny(tumor_pixels, sigma=2)
        binary_mask = binary_fill_holes(canny_mask)

        # Get tumor boundary points
        contours = find_contours(binary_mask, 0.5)
        if len(contours) == 0:
            print(f"No contours: {patient_id} | {modality}")
            continue

        largest = max(contours, key=len)

        # Compute convex hull
        try:
            hull = ConvexHull(largest)
        except Exception:
            print(f"Hull failed: {patient_id} | {modality}")
            continue

        # Plot
        fig, axes = plt.subplots(1, 2, figsize=(12, 6))

        # Original with convex hull overlay
        axes[0].imshow(raw_normalized, cmap='gray')
        axes[0].plot(largest[:, 1], largest[:, 0], 'r-', linewidth=1, label='Contour')
        for simplex in hull.simplices:
            axes[0].plot(largest[simplex, 1], largest[simplex, 0], 'b-', linewidth=1.5)
        axes[0].set_title('Convex Hull Overlay')
        axes[0].axis('off')

        # Tumor mask only
        axes[1].imshow(binary_mask, cmap='gray')
        for simplex in hull.simplices:
            axes[1].plot(largest[simplex, 1], largest[simplex, 0], 'b-', linewidth=1.5)
        axes[1].set_title('Convex Hull on Mask')
        axes[1].axis('off')

        plt.suptitle(f'Convex Hull — {patient_id} | {modality.upper()} | Slice {seg_slice_idx}', fontsize=14)
        plt.tight_layout()
        plt.savefig(f'{output_path}/{patient_id}_{modality}_convexhull.png', bbox_inches='tight')
        plt.show()
        plt.close()

    print(f"Done: {patient_id}")

print("\nAll patients done!")

# Deleting garbage data to free up RAM

In [ ]:
import gc
# This deletes the giant dictionary holding all 3D volumes
if 'images' in locals():
    del images
# Manually trigger garbage collection to empty the RAM
gc.collect()

# ***E-assignment 03***

### Step 1, 2 and 3: GLCM matrices, statistical descriptors and geometrical features

*Before we compute the matrices, its important to know that before this, i did assignment 3 by running the full assignment 2 and 1 pipeline, which caused my colab runtime to shoot upto approx 10 gb RAM usage out of 12, so i did assignment 3 again, but this time, i deleted garbage stuff that was saved, and in the following GLCM code, i run a loop which loads and scans data from drive, normalizes it, finds boondary and convex hull, computes glcm and saves the numbers into a list and immediatelty deletes heavy images from RAM. after this, a final_features.csv file will be saved to my drive, so for next sessions, i dont have to load all 3d volumes and run any canny or any such filters, i only need to run csv*


---



The Gray-Level Co-occurrence Matrix (GLCM) tracks how often pairs of pixels with specific values occur in a specific spatial relationship

how we'll proceed to calc GLCM matrices:
1. To compute the GLCM only for the tumor, we must first "fill" the boundary we found in Assignment 2 to create a solid binary mask. We then apply this mask to the original MRI slice.
2. Because MRI data often has high bit-depth (12-bit or 16-bit), we must first quantize the tumor pixels to a smaller number of gray levels (usually 8 or 16). If we use 256 levels, the matrix becomes too sparse and loses its statistical power.

Statistical descriptors mentioned in next step (energy, entropy, contrast) will also be calculated here, geometrical features (area, perim etc) will also be calculated in same code block and saved in csv



In [ ]:
import gc
# Using the variable from your notebook
my_path = dataset_path

patient_list = sorted([f for f in os.listdir(my_path) if f.startswith('BraTS')])
final_features = []
modalities_to_process = ['t1n', 't1c', 't2w', 't2f']

print(f"🚀 Extracting Texture and Geometric Features for {len(patient_list)} patients...")

for p_id in patient_list:
    patient_dir = os.path.join(my_path, p_id)
    if not os.path.isdir(patient_dir):
        continue

    # Load segmentation file for the current patient
    seg_file_name = [f for f in os.listdir(patient_dir) if 'seg' in f and (f.endswith('.nii.gz') or f.endswith('.nii'))]
    if not seg_file_name:
        print(f"Skipping {p_id}: No segmentation file found.")
        continue
    seg_data_path = os.path.join(patient_dir, seg_file_name[0])
    seg_data = nib.load(seg_data_path).get_fdata()

    # Find the slice with the largest tumor area
    tumor_pixel_counts = [np.sum(seg_data[:, :, i] > 0) for i in range(seg_data.shape[2])]
    seg_slice_idx = np.argmax(tumor_pixel_counts)

    # Check if the "best" slice actually contains a tumor
    if tumor_pixel_counts[seg_slice_idx] == 0:
        print(f"Skipping {p_id}: No tumor found in any slice of segmentation mask. No features extracted for this patient.")
        continue

    # Get the binary tumor mask for the selected slice
    patient_tumor_mask_slice = (seg_data[:, :, seg_slice_idx] > 0).astype(np.float32)

    for mod in modalities_to_process:
        file_name = f"{p_id}-{mod}.nii.gz"
        full_file_path = os.path.join(my_path, p_id, file_name)

        if not os.path.exists(full_file_path):
            print(f"Warning: Modality file {file_name} not found for {p_id}. Skipping.")
            continue

        # 1. Load specific image slice using the determined seg_slice_idx
        vol = nib.load(full_file_path).get_fdata()
        img_slice = vol[:, :, seg_slice_idx].astype(np.float32)
        del vol # Free up memory as the full volume is no longer needed

        # 2. Normalize the image slice
        norm = (img_slice - img_slice.min()) / (img_slice.max() - img_slice.min() + 1e-8)

        # Use the patient_tumor_mask_slice directly for feature extraction
        mask_for_features = (patient_tumor_mask_slice > 0).astype(int)

        # 3. GEOMETRIC FEATURES (Task 3)
        labeled_mask = label(mask_for_features)
        props = regionprops(labeled_mask)

        if props:
            # We assume the largest region is the tumor. If multiple regions, pick the biggest one.
            p = max(props, key=lambda prop: prop.area)
            # Ensure the selected region has a meaningful area
            if p.area == 0:
                print(f"No meaningful tumor region found for {p_id} - {mod} in selected slice. Skipping features.")
                continue

            area = p.area
            centroid = p.centroid # Returns (y, x)
            perimeter = p.perimeter
            circularity = (4 * np.pi * area) / (perimeter**2) if perimeter > 0 else 0

            # 4. TEXTURE FEATURES (Tasks 1 & 2)
            # Quantize only the relevant tumor intensity values for GLCM
            tumor_region_intensity = norm * mask_for_features
            quantized = (tumor_region_intensity * 15).astype(np.uint8)

            # Ensure non-zero pixels for GLCM calculation
            glcm_input_region = np.where(mask_for_features, quantized, 0)

            if np.sum(glcm_input_region > 0) < 2: # Requires at least two pixels to form co-occurrences
                 print(f"Not enough pixels for GLCM for {p_id} - {mod}. Assigning default texture values.")
                 energy, contrast, entropy = 0.0, 0.0, 0.0 # Assign default values if GLCM cannot be computed
            else:
                # GLCM levels should match the range of quantized values (0-15 -> 16 levels)
                glcm = graycomatrix(glcm_input_region, distances=[1], angles=[0], levels=16, symmetric=True, normed=True)

                energy = graycoprops(glcm, 'energy')[0, 0]
                contrast = graycoprops(glcm, 'contrast')[0, 0]
                entropy = -np.sum(glcm * np.log2(glcm + 1e-10)) # Add a small epsilon to avoid log(0)

            # 5. Store everything together
            final_features.append({
                'Patient': p_id,
                'Modality': mod.upper(),
                'Slice_Idx': seg_slice_idx,
                'Area': area,
                'Centroid_Y': centroid[0],
                'Centroid_X': centroid[1],
                'Perimeter': perimeter,
                'Circularity': circularity,
                'Energy': energy,
                'Contrast': contrast,
                'Entropy': entropy
            })
        else:
            print(f"No tumor region found after labeling for {p_id} - {mod} in selected slice. Skipping features.")

    gc.collect() # Collect garbage after processing each patient to manage memory

# Save one final CSV with ALL features
df_final = pd.DataFrame(final_features)
save_path = os.path.join(my_path, 'BraTS_Complete_Feature_Vector.csv')
df_final.to_csv(save_path, index=False)

print(f"✅ FINAL SUCCESS: All features saved to {save_path}")
display(df_final.head())

### Step 4: Traditional Classification

In Traditional Classification, we stop looking at pixels and start looking at numbers to make a diagnosis. Here is exactly what this step involves:

The classification step is where we teach the computer that:

* Malignant tumors usually have high entropy (chaotic texture) and low circularity (irregular shape).

* Benign tumors usually have low entropy (smooth texture) and high circularity (round shape).



---
what we need to do in this step:
1. For a computer to learn, it needs to know the "right answer" for the training data.we need a column in your CSV called Label.
Since we are using the BraTS dataset, the labels are usually provided in the metadata or based on the patient IDs (e.g., High-Grade Glioma vs. Low-Grade Glioma) but since is dataset mein labels arent provided, we will use unsupervised learning.
2. we will use K means clustering, for classifiying the tumors as malignant/benign, we will use the following philosophy:

* Cluster with High Entropy & Low Circularity: We label this as "Malignant" (Complex/Irregular).

* Cluster with Low Entropy & High Circularity: We label this as "Benign" (Smooth/Uniform).


In [ ]:
# 1. Load your complete feature vector
df = pd.read_csv('/content/drive/MyDrive/BraTS_Complete_Feature_Vector.csv')

# 2. Select the features for clustering
features = ['Area', 'Perimeter', 'Circularity', 'Energy', 'Contrast', 'Entropy']
X = df[features]

# 3. Scaling (CRITICAL: K-Means needs all numbers on the same scale)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4. Run K-Means to find 2 groups
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

# 5. Logic to assign "Malignant" vs "Benign"
# We assume the cluster with higher average Entropy is 'Malignant'
cluster_0_entropy = df[df['Cluster'] == 0]['Entropy'].mean()
cluster_1_entropy = df[df['Cluster'] == 1]['Entropy'].mean()

if cluster_1_entropy > cluster_0_entropy:
    df['Label'] = df['Cluster'].map({1: 'Malignant', 0: 'Benign'})
else:
    df['Label'] = df['Cluster'].map({0: 'Malignant', 1: 'Benign'})

# 6. Save the newly labeled dataset
df.to_csv('/content/drive/MyDrive/BraTS_Labeled_Features.csv', index=False)

print("✅ Labels assigned based on feature similarity!")
display(df[['Patient', 'Modality', 'Entropy', 'Circularity', 'Label']].head(10))

# ***E-final project***

final pipeline:

1.  Acquisition & Preprocessing: Load BraTS-PEDs dataset, bit-depth check, Gaussian/Mean/Median filtering, anti-aliasing
2. Segmentation: Canny edge detection, morphological cleaning
3. Specular Mitigation (bonus research challenge): Detect and remove bright specular spots that interfere with tumor detection
4. YOLO Preparation: Convert segmentation masks to YOLO bounding box format. Prepare train/test split
5. train yolo on prepared dataset
6. evaluation: prec, recall etc



---

since in previous assignments, we have alr done the first 2 steps, well start with specular mitigation

### Specular mitigation




what is it?

Specular mitigation is a technique used to remove unnaturally bright spots in an image, often called 'specular reflections' or 'hotspots.' In medical imaging, these can occur due to various factors, like a strong signal return from a particular tissue or even artifacts from the imaging process itself.


* In our code, we're identifying the top 1% of the brightest pixels in the image. These are likely to be specular reflections. Once identified, these bright pixels are replaced with the median intensity value of their surrounding pixels. This smoothes out the hotspot without losing too much important image information.

* Why the visual difference might be subtle: You might not always see a dramatic visual change because the reflections can be small or blend into already bright areas. The primary goal isn't to make the image look drastically different to the human eye, but to remove these subtle artifacts that could skew quantitative measurements or mislead machine learning models downstream.


In [ ]:
specular_removed = {}

for patient_id in sorted(images.keys()):
    specular_removed[patient_id] = {}
    patient_path = os.path.join(dataset_path, patient_id)

    # Load segmentation file for the current patient
    seg_file_name = [f for f in os.listdir(patient_path) if 'seg' in f and (f.endswith('.nii.gz') or f.endswith('.nii'))]
    if not seg_file_name:
        print(f"Skipping {patient_id}: No segmentation file found.")
        continue
    seg_data_path = os.path.join(patient_path, seg_file_name[0])
    seg_data = nib.load(seg_data_path).get_fdata()

    # Find the slice with the largest tumor area
    tumor_pixel_counts = [np.sum(seg_data[:, :, i] > 0) for i in range(seg_data.shape[2])]
    seg_slice_idx = np.argmax(tumor_pixel_counts)

    # Check if the "best" slice actually contains a tumor
    if tumor_pixel_counts[seg_slice_idx] == 0:
        print(f"Skipping {patient_id}: No tumor found in any slice of segmentation mask. No specular mitigation applied.")
        continue

    for modality in modalities_to_process:
        image_data_key = f"{patient_id}-{modality}"
        # Use the dynamically determined seg_slice_idx
        raw_slice = images[patient_id][image_data_key][:, :, seg_slice_idx]
        raw_normalized = (raw_slice - raw_slice.min()) / (raw_slice.max() - raw_slice.min() + 1e-8)

        # Detect specular regions (top 1% brightest pixels)
        threshold = np.percentile(raw_normalized, 99)
        specular_mask = raw_normalized > threshold

        # Replace specular regions with local median
        cleaned = raw_normalized.copy()
        cleaned[specular_mask] = median_filter(raw_normalized, size=5)[specular_mask]

        specular_removed[patient_id][modality] = cleaned

    print(f"Specular mitigation done for tumor slice {seg_slice_idx}: {patient_id}")

print("\nAll patients done!")

visualise specular mitigation before and after

In [ ]:
# Get a list of patient_ids that were processed for specular mitigation
processed_patient_ids = sorted(specular_removed.keys())

if not processed_patient_ids:
    print("No patients processed for specular mitigation.")
else:
    # Only show the first patient as an example, or loop through all if desired
    patient_id_to_display = processed_patient_ids[0]

    fig, axes = plt.subplots(len(modalities_to_process), 2, figsize=(10, 20))

    patient_path = os.path.join(dataset_path, patient_id_to_display)

    # Load segmentation file for the current patient to get seg_slice_idx
    seg_file_name = [f for f in os.listdir(patient_path) if 'seg' in f and (f.endswith('.nii.gz') or f.endswith('.nii'))]
    if not seg_file_name:
        print(f"No segmentation file found for {patient_id_to_display}. Cannot determine seg_slice_idx.")
    else:
        seg_data_path = os.path.join(patient_path, seg_file_name[0])
        seg_data = nib.load(seg_data_path).get_fdata()
        tumor_pixel_counts = [np.sum(seg_data[:, :, i] > 0) for i in range(seg_data.shape[2])]
        seg_slice_idx = np.argmax(tumor_pixel_counts)

        for i, modality in enumerate(modalities_to_process):
            image_data_key = f"{patient_id_to_display}-{modality}"

            # Retrieve the original raw slice using the determined seg_slice_idx
            raw_slice = images[patient_id_to_display][image_data_key][:, :, seg_slice_idx]
            raw_normalized = (raw_slice - raw_slice.min()) / (raw_slice.max() - raw_slice.min() + 1e-8)

            axes[i][0].imshow(raw_normalized, cmap='gray')
            axes[i][0].set_title(f'{modality.upper()} — Before Specular Mitigation')
            axes[i][0].axis('off')

            if modality in specular_removed[patient_id_to_display]:
                axes[i][1].imshow(specular_removed[patient_id_to_display][modality], cmap='gray')
                axes[i][1].set_title(f'{modality.upper()} — After Specular Mitigation')
            else:
                axes[i][1].set_title(f'{modality.upper()} — Not Processed')
                axes[i][1].axis('off') # Turn off axis if no image to show

            axes[i][1].axis('off')

        plt.suptitle(f'Specular Mitigation — {patient_id_to_display} (Slice {seg_slice_idx})', fontsize=14)
        plt.tight_layout()
        plt.show()

### Applying YOLO for tumor localisation

In [ ]:
yolo_path = '/content/drive/MyDrive/yolo_dataset'
os.makedirs(f'{yolo_path}/images/train', exist_ok=True)
os.makedirs(f'{yolo_path}/images/val', exist_ok=True)
os.makedirs(f'{yolo_path}/labels/train', exist_ok=True)
os.makedirs(f'{yolo_path}/labels/val', exist_ok=True)

def get_yolo_bbox(binary_mask):
    """Convert binary mask to YOLO format bounding box"""
    props = regionprops(label(binary_mask))
    if len(props) == 0:
        return None
    # Get largest region
    largest = max(props, key=lambda r: r.area)
    minr, minc, maxr, maxc = largest.bbox
    h, w = binary_mask.shape

    # YOLO format: class x_center y_center width height (all normalized 0-1)
    x_center = ((minc + maxc) / 2) / w
    y_center = ((minr + maxr) / 2) / h
    width = (maxc - minc) / w
    height = (maxr - minr) / h

    return x_center, y_center, width, height

img_count = 0
patients = sorted(images.keys())
train_patients = patients[:20]
val_patients = patients[20:]

for patient_id in patients:
    split = 'train' if patient_id in train_patients else 'val'
    patient_path = os.path.join(dataset_path, patient_id)

    # Load seg file
    seg_file = [f for f in os.listdir(patient_path) if 'seg' in f][0]
    seg_data = nib.load(os.path.join(patient_path, seg_file)).get_fdata()

    # Find best slice
    seg_slice_idx = np.argmax([np.sum(seg_data[:, :, i] > 0) for i in range(seg_data.shape[2])])
    seg_slice = seg_data[:, :, seg_slice_idx]
    tumor_mask = (seg_slice > 0)

    for modality in modalities_to_process:
        image_data_key = f'{patient_id}-{modality}'
        raw_slice = images[patient_id][image_data_key][:, :, seg_slice_idx]
        raw_normalized = (raw_slice - raw_slice.min()) / (raw_slice.max() - raw_slice.min() + 1e-8)

        # Specular mitigation
        threshold = np.percentile(raw_normalized, 99)
        specular_mask = raw_normalized > threshold
        cleaned = raw_normalized.copy()
        cleaned[specular_mask] = median_filter(raw_normalized, size=5)[specular_mask]

        # Get bounding box from seg mask
        bbox = get_yolo_bbox(tumor_mask)
        if bbox is None:
            continue

        # Save image
        img_uint8 = (cleaned * 255).astype(np.uint8)
        img_name = f'{patient_id}_{modality}'
        cv2.imwrite(f'{yolo_path}/images/{split}/{img_name}.png', img_uint8)

        # Save label
        with open(f'{yolo_path}/labels/{split}/{img_name}.txt', 'w') as f:
            f.write(f'0 {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f}\n')

        img_count += 1

print(f"Total images prepared: {img_count}")
print(f"Train: {len(train_patients) * len(modalities_to_process)} images")
print(f"Val: {len(val_patients) * len(modalities_to_process)} images")

yolo config file

In [ ]:
yaml_content = f"""
path: {yolo_path}
train: images/train
val: images/val

nc: 1
names: ['tumor']
"""

with open(f'{yolo_path}/dataset.yaml', 'w') as f:
    f.write(yaml_content)

print("YOLO config file created!")
print(yaml_content)

train yolo

In [ ]:
# Load pretrained YOLOv8 small model
model = YOLO('yolov8s.pt')

# Train
results = model.train(
    data=f'{yolo_path}/dataset.yaml',
    epochs=50,
    imgsz=240,
    batch=8,
    name='brain_tumor_yolo',
    project='/content/drive/MyDrive/yolo_results',
    device=0  # GPU
)

print("Training done!")

In [ ]:
# Load the trained model
model = YOLO('/content/drive/MyDrive/yolo_results/brain_tumor_yolo/weights/best.pt')

# Run validation
metrics = model.val(
    data=f'{yolo_path}/dataset.yaml',
    device=0
)

# Print results
print("\n========== EVALUATION RESULTS ==========")
print(f"Precision:  {metrics.box.p[0]:.4f}")
print(f"Recall:     {metrics.box.r[0]:.4f}")
print(f"mAP@0.5:    {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print("=========================================")

In [ ]:
# Plot confusion matrix
model.val(
    data=f'{yolo_path}/dataset.yaml',
    device=0,
    plots=True,
    save_dir='/content/drive/MyDrive/yolo_results/evaluation'
)

# Visualize predictions on validation images
val_images = sorted(os.listdir(f'{yolo_path}/images/val'))[:5]  # first 5

fig, axes = plt.subplots(1, 5, figsize=(25, 5))

for i, img_name in enumerate(val_images):
    img_path = f'{yolo_path}/images/val/{img_name}'
    results = model.predict(img_path, device=0, verbose=False)

    # Plot result
    result_img = results[0].plot()
    result_img = cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)

    axes[i].imshow(result_img, cmap='gray')
    axes[i].set_title(img_name.replace('.png', ''))
    axes[i].axis('off')

plt.suptitle('YOLO Predictions on Validation Set', fontsize=14)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/yolo_results/evaluation/predictions.png', bbox_inches='tight')
plt.show()
print("Done!")

# Extra

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from skimage.feature import canny
from skimage.filters import sobel
from skimage.morphology import disk, dilation, closing
from skimage.measure import find_contours
from scipy.spatial import ConvexHull
from IPython.display import display, HTML

# 1. Helper Function for Chain Code (8-connectivity)
def get_chain_code(contour):
    lookup = {(0, 1): 0, (1, 1): 7, (1, 0): 6, (1, -1): 5, (0, -1): 4, (-1, -1): 3, (-1, 0): 2, (-1, 1): 1}
    chain_code = []
    for i in range(len(contour) - 1):
        diff = (int(round(contour[i+1][0] - contour[i][0])), int(round(contour[i+1][1] - contour[i][1])))
        if diff in lookup: chain_code.append(lookup[diff])
    return "".join(map(str, chain_code))

# 2. Configuration
sample_patients = sorted(list(images.keys()))[:5] # First 5 for the table
modalities = ['t1n', 't1c', 't2w', 't2f']
report_data = []

print("🔄 Re-generating deliverables for the report...")

for p_idx, patient_id in enumerate(sample_patients):
    # Only show visual overlays for the first patient to save space
    if p_idx == 0:
        fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    for m_idx, mod in enumerate(modalities):
        mod_key = next((k for k in images[patient_id].keys() if mod in k), None)
        if not mod_key: continue

        # --- A. SEGMENTATION & CLEANING ---
        raw_slice = images[patient_id][mod_key][:, :, 77]
        raw_norm = (raw_slice - raw_slice.min()) / (raw_slice.max() - raw_slice.min() + 1e-8)
        # Week 2: Edge Detection
        edges = canny(raw_norm, sigma=2)
        # Week 3: Morphological Cleaning (Closing)
        closed = closing(edges, disk(2))

        # --- B. BOUNDARY REPRESENTATION (Week 4) ---
        contours = find_contours(closed, 0.5)
        if contours:
            main_contour = max(contours, key=len)
            c_code = get_chain_code(main_contour)

            # Save data for the Table
            report_data.append({
                "Patient ID": patient_id,
                "Modality": mod.upper(),
                "Boundary Px": len(c_code),
                "Chain Code Snippet (First 20 digits)": c_code[:20] + "..."
            })

            # --- C. COMPUTATIONAL GEOMETRY (Week 5) ---
            if p_idx == 0: # Visualize for the first patient
                hull = ConvexHull(main_contour)
                hull_pts = main_contour[hull.vertices]
                hull_plot = np.vstack((hull_pts, hull_pts[0]))

                axes[m_idx].imshow(raw_norm, cmap='gray')
                axes[m_idx].plot(hull_plot[:, 1], hull_plot[:, 0], 'cyan', linewidth=2)
                axes[m_idx].set_title(f"{mod.upper()} - Hull Overlay")
                axes[m_idx].axis('off')

    if p_idx == 0:
        plt.suptitle(f"Deliverable: Convex Hull Overlays on Original MRI Data ({patient_id})", fontsize=14)
        plt.show()

# 3. DISPLAY THE CHAIN CODE TABLE
print("\n✅ Deliverable: Chain Code Table (8-connectivity)")
df_report = pd.DataFrame(report_data)
display(HTML(df_report.to_html(index=False)))